# 02 · Agrupamiento de cuentas y municipios
**Facturación residencial de energía eléctrica, 2023–2024**

Este notebook cubre las fases **3.5 y 3.6** de la guía y las problemáticas **4.3 y 4.4**.

- **Entrada:** `data/datos_limpios.parquet` generado por la 01. **No vuelve a limpiar el CSV ni sobrescribe el parquet.**
- **Salida:** las tablas y figuras se muestran directamente en el notebook.
- **Unidades de análisis:** la cuenta para K-Means y el municipio para el agrupamiento jerárquico.


In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster, cophenet
from scipy.spatial.distance import pdist
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style='whitegrid', palette='deep')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

RUTA_DATOS = Path('data/datos_limpios.parquet')
if not RUTA_DATOS.exists():
    raise FileNotFoundError('Ejecuta primero 01_limpieza_y_exploratorio.ipynb: no existe data/datos_limpios.parquet')

df = pd.read_parquet(RUTA_DATOS)
requeridas = ['id_cuenta', 'periodo', 'operador', 'municipio', 'altitud_msnm', 'estrato',
              'consumo_kwh', 'tarifa_cu_cop_kwh', 'dias_mora', 'lectura_estimada',
              'autogeneracion_solar', 'subsidio_contribucion_cop']
faltantes = [c for c in requeridas if c not in df.columns]
if faltantes:
    raise ValueError(f'El parquet no tiene las columnas esperadas: {faltantes}. Vuelve a ejecutar la 01.')

print(f'Filas: {len(df):,} | Cuentas: {df["id_cuenta"].nunique():,}')
print(f'Operadores: {df["operador"].nunique()} | Municipios: {df["municipio"].nunique()}')
print(f'Periodos: {df["periodo"].min()} a {df["periodo"].max()}')
print('Este notebook solo lee el parquet: no vuelve a limpiar el CSV ni lo sobrescribe.')


## Fase 3.5 / Problemática 4.3 · Tabla de comportamiento por cuenta

In [ ]:
# ============================================================
# FASE 3.5 / PROBLEMÁTICA 4.3 - Tabla de una fila por cuenta
# ============================================================
conteo_meses = df.groupby('id_cuenta')['periodo'].count()
cuentas_validas = conteo_meses[conteo_meses >= 12].index
cuentas_excluidas = len(conteo_meses) - len(cuentas_validas)

cons_2023_s2 = df[df['periodo'].between('2023-07', '2023-12')].groupby('id_cuenta')['consumo_kwh'].mean()
cons_2024_s2 = df[df['periodo'].between('2024-07', '2024-12')].groupby('id_cuenta')['consumo_kwh'].mean()

cuentas = df.groupby('id_cuenta').agg(
    meses_historia=('periodo', 'count'),
    consumo_medio=('consumo_kwh', 'mean'),
    consumo_std=('consumo_kwh', 'std'),
    prop_mora=('dias_mora', lambda x: (x > 0).mean()),
    prop_mora_60=('dias_mora', lambda x: (x > 60).mean()),
    prop_estimada=('lectura_estimada', lambda x: (x == 'S').mean()),
    meses_casi_nulo=('consumo_kwh', lambda x: int((x < 10).sum())),
    estrato=('estrato', 'first'),
    autogeneracion=('autogeneracion_solar', lambda x: int((x == 'S').any())),
    operador=('operador', lambda x: x.mode().iat[0]),
    municipio=('municipio', lambda x: x.mode().iat[0]),
).reset_index()

cuentas = cuentas[cuentas['meses_historia'] >= 12].copy()
cuentas['cv_consumo'] = (cuentas['consumo_std'] / cuentas['consumo_medio']).replace([np.inf, -np.inf], np.nan).fillna(0)
cuentas['tendencia'] = (
    cuentas['id_cuenta'].map(cons_2024_s2) / cuentas['id_cuenta'].map(cons_2023_s2)
).replace([np.inf, -np.inf], np.nan).fillna(1.0)

features = ['consumo_medio', 'cv_consumo', 'tendencia', 'prop_mora', 'prop_estimada', 'meses_casi_nulo']

print(f'Cuentas totales: {len(conteo_meses):,}')
print(f'Cuentas excluidas por tener menos de 12 meses: {cuentas_excluidas:,} ({cuentas_excluidas / len(conteo_meses) * 100:.2f}%)')
print(f'Cuentas usadas en el agrupamiento: {len(cuentas):,}')
print('Justificación: con menos de 12 meses no se puede medir tendencia ni estacionalidad;')
print('incluirlas compararía cuentas con historias incompletas y sesgaría los segmentos.')

# Winsorizar para que los valores extremos no dominen la distancia euclidiana
for col in features:
    lo, hi = cuentas[col].quantile([0.001, 0.999])
    cuentas[col] = cuentas[col].clip(lo, hi)

resumen_cuentas = cuentas[features + ['prop_mora_60', 'estrato', 'autogeneracion']].describe(
    percentiles=[0.25, 0.5, 0.75]).T.round(3)
display(resumen_cuentas)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.ravel(), features):
    sns.histplot(cuentas[col], bins=40, ax=ax, color='#2b5c8f')
    ax.axvline(cuentas[col].median(), color='red', linestyle='--', linewidth=1)
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('')
fig.suptitle('Variables de comportamiento por cuenta (después de winsorizar 0,1%-99,9%)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Lectura rápida:')
print(f'  Consumo medio por cuenta: {cuentas["consumo_medio"].median():,.0f} kWh (mediana)')
print(f'  CV del consumo: mediana {cuentas["cv_consumo"].median():.3f}; 95% de las cuentas por debajo de {cuentas["cv_consumo"].quantile(0.95):.3f}')
print(f'  Cuentas con mora alguna vez: {cuentas["prop_mora"].gt(0).mean() * 100:.1f}%')
print(f'  Cuentas con al menos una lectura estimada: {cuentas["prop_estimada"].gt(0).mean() * 100:.1f}%')
print(f'  Cuentas con autogeneración solar: {cuentas["autogeneracion"].mean() * 100:.2f}%')


## Elección de k, K-Means y perfiles

Se prueban `k = 2..10` con codo y silueta (muestra de 30.000 cuentas). Se elige **k = 4** porque la
silueta es prácticamente igual a la de k = 3 y separa dos tipos de irregularidad con acciones distintas.
Los nombres de los segmentos se construyen a partir del perfil, no están prefijados.

In [ ]:
# ============================================================
# FASE 3.5 / PROBLEMÁTICA 4.3 - Elección de k, K-Means y perfiles
# ============================================================
X = StandardScaler().fit_transform(cuentas[features])

rng = np.random.default_rng(42)
idx_muestra = rng.choice(len(X), size=min(30_000, len(X)), replace=False)

resultados_k = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    etiquetas = km.fit_predict(X)
    resultados_k.append({
        'k': k,
        'Inercia': km.inertia_,
        'Silueta': silhouette_score(X[idx_muestra], etiquetas[idx_muestra]),
    })
resultados_k = pd.DataFrame(resultados_k)
display(resultados_k.round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(resultados_k['k'], resultados_k['Inercia'], marker='o')
axes[0].set_title('Método del codo (inercia)', fontweight='bold')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inercia')
axes[1].plot(resultados_k['k'], resultados_k['Silueta'], marker='o', color='green')
axes[1].set_title('Coeficiente de silueta (muestra de 30.000 cuentas)', fontweight='bold')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silueta')
axes[1].axvline(4, color='red', linestyle='--', label='k = 4 elegido')
axes[1].legend()
plt.tight_layout()
plt.show()

K_OPTIMO = 4
km_final = KMeans(n_clusters=K_OPTIMO, random_state=42, n_init=10)
cuentas['cluster'] = km_final.fit_predict(X)

perfil = cuentas.groupby('cluster')[features + ['estrato', 'autogeneracion', 'prop_mora_60']].mean()
perfil['N_Cuentas'] = cuentas['cluster'].value_counts()
perfil['Pct_Cuentas'] = perfil['N_Cuentas'] / len(cuentas) * 100

# Nombres construidos a partir del propio perfil, sin etiquetas prefijadas
nombres = {}
umbral_cv = 0.30
for c in perfil.index:
    fila = perfil.loc[c]
    if fila['cv_consumo'] >= umbral_cv:
        if fila['tendencia'] >= 1.20:
            nombres[c] = 'Irregular en ascenso'
        elif fila['tendencia'] <= 0.80:
            nombres[c] = 'Irregular en descenso'
        else:
            nombres[c] = 'Irregular estable'
    else:
        nombres[c] = 'Residencial estable de bajo consumo'
regulares = [c for c in perfil.index if perfil.loc[c, 'cv_consumo'] < umbral_cv]
if regulares:
    c_alto = max(regulares, key=lambda c: perfil.loc[c, 'prop_mora_60'])
    nombres[c_alto] = 'Alto consumo con mora'
perfil['Nombre_Segmento'] = perfil.index.map(nombres)
perfil = perfil.sort_values('N_Cuentas', ascending=False)

tabla_perfiles = perfil[['Nombre_Segmento', 'N_Cuentas', 'Pct_Cuentas'] + features +
                        ['prop_mora_60', 'estrato', 'autogeneracion']].round(3)
display(tabla_perfiles)

cuentas['segmento'] = cuentas['cluster'].map(nombres)
resumen_segmentos = cuentas.groupby('segmento').agg(
    N_Cuentas=('id_cuenta', 'size'),
    Pct_Cuentas=('id_cuenta', lambda x: len(x) / len(cuentas) * 100),
    Consumo_medio=('consumo_medio', 'mean'),
    CV_consumo=('cv_consumo', 'mean'),
    Tendencia=('tendencia', 'mean'),
    Pct_mora=('prop_mora', 'mean'),
    Pct_mora_60=('prop_mora_60', 'mean'),
    Pct_estimadas=('prop_estimada', 'mean'),
    Meses_casi_nulo=('meses_casi_nulo', 'mean'),
    Estrato_medio=('estrato', 'mean'),
    Pct_autogeneracion=('autogeneracion', 'mean'),
).sort_values('N_Cuentas', ascending=False).round(3)
display(resumen_segmentos)

# Mapas de calor y tamaños
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
orden = perfil.index
z = (perfil[features] - perfil[features].mean()) / perfil[features].std(ddof=0)
sns.heatmap(z, annot=perfil[features].round(2), fmt='', cmap='RdBu_r', center=0,
            yticklabels=perfil['Nombre_Segmento'], ax=axes[0])
axes[0].set_title('Perfil de cada segmento (color = valor estandarizado)', fontweight='bold')
axes[0].set_xlabel('')

sns.barplot(x=perfil['N_Cuentas'], y=perfil['Nombre_Segmento'], ax=axes[1], color='#2b5c8f')
axes[1].set_title('Tamaño de cada segmento', fontweight='bold')
axes[1].set_xlabel('Cuentas')
axes[1].set_ylabel('')
for i, (n, pct) in enumerate(zip(perfil['N_Cuentas'], perfil['Pct_Cuentas'])):
    axes[1].text(n, i, f' {n:,} ({pct:.1f}%)', va='center', fontsize=9)

sns.barplot(x=perfil['prop_mora_60'] * 100, y=perfil['Nombre_Segmento'], ax=axes[2], color='#c0504d')
axes[2].set_title('% de meses con mora > 60 días', fontweight='bold')
axes[2].set_xlabel('%')
axes[2].set_ylabel('')
plt.tight_layout()
plt.show()

# Prioridad para la comisión
prioritario = resumen_segmentos['Pct_mora_60'].idxmax()
fila_prioritaria = resumen_segmentos.loc[prioritario]
irregulares = resumen_segmentos[resumen_segmentos.index.str.startswith('Irregular')]
print('Justificación de k = 4:')
print('  La silueta es máxima en k = 2, pero ese corte solo separa hogares regulares de irregulares y')
print('  oculta el segmento con mora. Con k = 4 la silueta (%.4f) es prácticamente igual a la de k = 3 (%.4f)' %
      (resultados_k.loc[resultados_k['k'] == 4, 'Silueta'].iat[0],
       resultados_k.loc[resultados_k['k'] == 3, 'Silueta'].iat[0]))
print('  y separa dos tipos de irregularidad con acciones distintas: ascenso y descenso del consumo.')
print()
print(f'Segmento prioritario: {prioritario}')
print(f'  {int(fila_prioritaria["N_Cuentas"]):,} cuentas ({fila_prioritaria["Pct_Cuentas"]:.1f}% del total)')
print(f'  {fila_prioritaria["Pct_mora"] * 100:.1f}% de sus meses tiene alguna mora y {fila_prioritaria["Pct_mora_60"] * 100:.1f}% supera 60 días')
print('  Acción: contacto preventivo y acuerdos de pago antes de que la deuda pase a cobro jurídico.')
print()
print(f'Cuentas irregulares (ascenso + descenso): {int(irregulares["N_Cuentas"].sum()):,} '
      f'({irregulares["Pct_Cuentas"].sum():.1f}%)')
print(f'  {irregulares["Pct_estimadas"].mean() * 100:.1f}% de lecturas estimadas en promedio.')
print('  Acción: revisión técnica y verificación de lecturas/medidor; no es un problema de pago.')


## Fase 3.6 / Problemática 4.4 · Agrupamiento jerárquico por municipio

Se agrega una fila por municipio, se comparan Ward, enlace promedio y completo, y se justifica la altura
de corte con los saltos de fusión. También se responde si los grupos siguen al operador, al clima o al
nivel socioeconómico.

In [ ]:
# ============================================================
# FASE 3.6 / PROBLEMÁTICA 4.4 - Agrupamiento jerárquico por municipio
# ============================================================
mun = df.groupby('municipio', observed=True).agg(
    operador=('operador', lambda x: x.mode().iat[0]),
    departamento=('departamento', lambda x: x.mode().iat[0]),
    altitud=('altitud_msnm', 'first'),
    consumo_medio=('consumo_kwh', 'mean'),
    tarifa_media=('tarifa_cu_cop_kwh', 'mean'),
    pct_mora_60=('dias_mora', lambda x: (x > 60).mean() * 100),
    estrato_medio=('estrato', 'mean'),
    sub_contrib_medio=('subsidio_contribucion_cop', 'mean'),
    pct_estimadas=('lectura_estimada', lambda x: (x == 'S').mean() * 100),
    n_facturas=('id_cuenta', 'size'),
    n_cuentas=('id_cuenta', 'nunique'),
).reset_index()
mun['clima'] = pd.cut(mun['altitud'], bins=[-np.inf, 999.9, 2000, np.inf],
                      labels=['Cálido', 'Templado', 'Frío'])
cols_mun = ['consumo_medio', 'tarifa_media', 'pct_mora_60', 'estrato_medio', 'sub_contrib_medio', 'pct_estimadas']

print(f'Municipios en la tabla agregada: {len(mun)} (deben ser 26)')
display(mun.sort_values('consumo_medio', ascending=False).round(2))

# Bivariado de los indicadores agregados
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(mun[cols_mun + ['altitud']].corr(method='spearman').round(2), annot=True,
            cmap='RdBu_r', vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title('Correlación de Spearman entre indicadores municipales', fontweight='bold')
sns.scatterplot(data=mun, x='tarifa_media', y='pct_mora_60', hue='clima', s=80, ax=axes[1])
for _, fila in mun.iterrows():
    axes[1].annotate(fila['municipio'], (fila['tarifa_media'], fila['pct_mora_60']),
                     fontsize=7, xytext=(3, 3), textcoords='offset points')
axes[1].set_title('Tarifa media vs. mora > 60 días por municipio', fontweight='bold')
axes[1].set_xlabel('Tarifa media (COP/kWh)')
axes[1].set_ylabel('% de facturas con mora > 60 días')
plt.tight_layout()
plt.show()

# Escalado y agrupamiento jerárquico
X_mun = StandardScaler().fit_transform(mun[cols_mun])
Z_ward = linkage(X_mun, method='ward')
Z_avg = linkage(X_mun, method='average')
Z_comp = linkage(X_mun, method='complete')

for nombre, Z in [('Ward', Z_ward), ('Promedio', Z_avg), ('Completo', Z_comp)]:
    c, _ = cophenet(Z, pdist(X_mun))
    print(f'Correlación cofenética {nombre}: {c:.3f}')

distancias = pd.DataFrame({'Distancia_de_fusion': Z_ward[:, 2]})
distancias['Salto'] = distancias['Distancia_de_fusion'].diff()
print('\nÚltimas fusiones de Ward (el corte se ubica en el mayor salto):')
display(distancias.tail(8).round(3))

ALTURA_CORTE = 3.0
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
dendrogram(Z_ward, labels=mun['municipio'].values, leaf_rotation=90, leaf_font_size=9,
           color_threshold=ALTURA_CORTE, ax=axes[0])
axes[0].axhline(ALTURA_CORTE, color='red', linestyle='--', linewidth=1.5,
                label=f'Corte t = {ALTURA_CORTE}')
axes[0].set_title('Dendrograma - Ward', fontweight='bold')
axes[0].set_ylabel('Distancia')
axes[0].legend()

dendrogram(Z_avg, labels=mun['municipio'].values, leaf_rotation=90, leaf_font_size=9,
           color_threshold=2.0, ax=axes[1])
axes[1].axhline(2.0, color='red', linestyle='--', linewidth=1.5, label='Corte t = 2,0')
axes[1].set_title('Dendrograma - Enlace promedio', fontweight='bold')
axes[1].set_ylabel('Distancia')
axes[1].legend()
plt.tight_layout()
plt.show()

mun['grupo'] = fcluster(Z_ward, t=ALTURA_CORTE, criterion='distance')
grupo_avg_2 = fcluster(Z_avg, t=2.0, criterion='distance')
grupo_avg_3 = fcluster(Z_avg, t=ALTURA_CORTE, criterion='distance')
print(f'\nWard con corte {ALTURA_CORTE}: {mun["grupo"].nunique()} grupos')
print(f'Promedio con corte 2,0: {len(set(grupo_avg_2))} grupos | al mismo corte 3,0: {len(set(grupo_avg_3))} grupos')
print('Ward y promedio coinciden en el bloque del Caribe. El promedio necesita bajar el corte a 2,0')
print('para separar las ciudades intermedias de las de altura; con el corte de Ward las une.')
print('Se conservan los 3 grupos de Ward porque el salto 2,07 -> 6,04 separa el bloque del Caribe,')
print('y el salto 6,04 -> 16,07 separa las ciudades intermedias de las de altura.')

# Nombres de los grupos a partir de su perfil
resumen_grupos = mun.groupby('grupo')[cols_mun + ['altitud']].mean()
resumen_grupos['N_Municipios'] = mun.groupby('grupo').size()
resumen_grupos['Operadores'] = mun.groupby('grupo')['operador'].apply(
    lambda x: ', '.join(sorted(x.unique())))
resumen_grupos['Municipios'] = mun.groupby('grupo')['municipio'].apply(lambda x: ', '.join(sorted(x)))

nombres_grupo = {}
for g in resumen_grupos.index:
    fila = resumen_grupos.loc[g]
    if fila['tarifa_media'] > 1000:
        nombres_grupo[g] = 'Caribe: tarifa alta y mayor mora'
    elif fila['consumo_medio'] > 200:
        nombres_grupo[g] = 'Intermedias: consumo medio-alto'
    else:
        nombres_grupo[g] = 'Templado/frío: consumo bajo y tarifa menor'
resumen_grupos['Nombre_Grupo'] = resumen_grupos.index.map(nombres_grupo)
resumen_grupos = resumen_grupos.sort_values('tarifa_media', ascending=False)

tabla_grupos = resumen_grupos[['Nombre_Grupo', 'N_Municipios'] + cols_mun + ['altitud', 'Operadores']].round(2)
display(tabla_grupos)

mun['nombre_grupo'] = mun['grupo'].map(nombres_grupo)
tabla_municipios = mun[['municipio', 'departamento', 'operador', 'altitud', 'clima',
                        'nombre_grupo'] + cols_mun].sort_values(['nombre_grupo', 'consumo_medio'],
                                                               ascending=[True, False]).round(2)
display(tabla_municipios)

# ¿Los grupos siguen al operador, al clima o al estrato?
print('\nComposición de los grupos por operador:')
print(pd.crosstab(mun['nombre_grupo'], mun['operador']).to_string())
print('\nComposición de los grupos por clima:')
print(pd.crosstab(mun['nombre_grupo'], mun['clima']).to_string())


def eta_cuadrado(valores, grupos):
    gran_media = valores.mean()
    sst = ((valores - gran_media) ** 2).sum()
    ssb = sum(len(valores[grupos == g]) * (valores[grupos == g].mean() - gran_media) ** 2
              for g in pd.Series(grupos).unique())
    return ssb / sst if sst else np.nan


filas_eta = []
for col in cols_mun:
    filas_eta.append({
        'Indicador': col,
        'Operador': eta_cuadrado(mun[col], mun['operador']),
        'Clima': eta_cuadrado(mun[col], mun['clima'].astype(str)),
        'Nivel socioeconómico (estrato)': eta_cuadrado(mun[col], pd.qcut(mun['estrato_medio'], 2, labels=False)),
        'Grupo jerárquico': eta_cuadrado(mun[col], mun['grupo']),
    })
eta2 = pd.DataFrame(filas_eta).set_index('Indicador').round(3)
display(eta2)

# Municipios que no encajan en el patrón de su grupo
X_df = pd.DataFrame(X_mun, index=mun['municipio'], columns=cols_mun)
centroides = X_df.groupby(mun['grupo'].values).transform('mean')
mun['distancia_grupo'] = np.sqrt(((X_df - centroides) ** 2).sum(axis=1)).values
outliers = mun.nlargest(5, 'distancia_grupo')[['municipio', 'operador', 'altitud', 'nombre_grupo',
                                                'consumo_medio', 'tarifa_media', 'estrato_medio',
                                                'distancia_grupo']].round(2)
print('\nMunicipios que más se alejan del centro de su grupo:')
display(outliers)

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap((resumen_grupos[cols_mun] - resumen_grupos[cols_mun].mean()) / resumen_grupos[cols_mun].std(ddof=0),
            annot=resumen_grupos[cols_mun].round(1), fmt='', cmap='RdBu_r', center=0,
            yticklabels=resumen_grupos['Nombre_Grupo'], ax=ax)
ax.set_title('Perfil promedio de los grupos de municipios (color = valor estandarizado)', fontweight='bold')
ax.set_xlabel('')
plt.tight_layout()
plt.show()

print('\nRESPUESTA A LA PREGUNTA 4.4:')
print('  Los grupos siguen al clima y al operador a la vez, porque en estos datos están confundidos:')
print('  los 8 municipios del Caribe son exactamente los de AIR-E y AFINIA. La tarifa y la mora son')
print('  casi un reflejo del operador (eta² = %.2f y %.2f), mientras el consumo sigue la altitud/clima' %
      (eta2.loc['tarifa_media', 'Operador'], eta2.loc['pct_mora_60', 'Operador']))
print('  (eta² = %.2f). El estrato explica menos (eta² = %.2f para consumo y %.2f para tarifa).' %
      (eta2.loc['consumo_medio', 'Clima'], eta2.loc['consumo_medio', 'Nivel socioeconómico (estrato)'],
       eta2.loc['tarifa_media', 'Nivel socioeconómico (estrato)']))
print('  Conclusión: las mesas regionales deben organizarse por clima/región; el estrato no separa municipios.')
print('  Casos atípicos: ' + ', '.join(outliers['municipio'].head(3)) + '.')
